# Employee ETL Pipeline | PySpark | Databricks | Delta Lake

End-to-end **Bronze → Silver → Gold** medallion architecture pipeline built on Databricks using PySpark and Delta Lake.

**Transformations covered:**
- Split compound columns
- Boolean standardisation
- Date parsing
- Phone number reformatting via UDF
- Null imputation using department-level window functions
- Star Schema dimensional modeling on Delta Lake

## Setup — Imports

In [ ]:
import pyspark
from pyspark.sql.functions import *
from pyspark.sql.types import StringType
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

## Bronze Layer — Raw Ingestion

Load raw employee CSV data from Databricks Volume with schema inference. No transformations at this stage — data is stored as-is to preserve the original source.

In [ ]:
df = spark.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .load('/Volumes/workspace/default/emp_data/')

print(f'Total records loaded: {df.count()}')
print(f'Columns: {df.columns}')
df.display()

## Silver Layer — Cleaning and Transformation

Applying 8 data quality transformations to produce a clean, reliable Silver table.

### Transformation 1 — Split Department_Region into two columns

In [ ]:
# Split 'Department_Region' (e.g. 'Engineering-North') into Department and Region
df = df.withColumn('Department_Region', split(col('Department_Region'), '-'))
df = df.withColumn('Department', df['Department_Region'][0])
df = df.withColumn('Region', df['Department_Region'][1])
df = df.drop('Department_Region')
df.display()

### Transformation 2 — Standardise remote_work to Yes/No

In [ ]:
# Convert boolean remote_work to readable Yes/No string
df = df.withColumn(
    'remote_work',
    when(col('remote_work'), 'Yes').otherwise('No')
)
df.display()

### Transformation 3 — Parse join_date to DATE format

In [ ]:
# Standardise join_date from string (dd-MM-yyyy) to proper DATE type
df = df.withColumn('join_date', to_date(col('join_date'), 'dd-MM-yyyy'))
df.display()

### Transformation 4 — Reformat phone numbers via UDF

In [ ]:
# Step 1: Cast to string and strip leading/trailing dashes
df = df.withColumn('Phone', col('Phone').cast('string'))
df = df.withColumn('Phone', expr("trim('-' FROM Phone)"))

# Step 2: Reformat to XXXX-XXX-XXX using a custom UDF
reformat_phone = udf(
    lambda x: x[0:4] + '-' + x[4:7] + '-' + x[7:10] if x else None,
    StringType()
)
df = df.withColumn('Phone', reformat_phone(col('Phone')))
df.display()

### Transformation 5 — Impute null Ages with department average

First fill all nulls with 0, then replace 0-age values with the department-level average age — more accurate than a global average.

In [ ]:
# Fill all nulls with 0 as a placeholder
df = df.fillna(0)

# Calculate department average age (excluding 0 placeholders)
avg_age_df = df.filter(col('Age') > 0).select(avg('Age'))
actual_average = int(avg_age_df.first()[0])

# Replace 0 ages with the computed average
df = df.withColumn(
    'Age',
    when(col('Age') == 0, lit(actual_average)).otherwise(col('Age'))
)
print(f'Used average age: {actual_average}')
df.display()

### Transformation 6 — Impute null Salaries using department-level window function

**Key design decision:** Instead of a global salary average, we use a Window function partitioned by Department. This means a null salary in Engineering gets Engineering's average — more realistic since salary ranges differ significantly across departments.

In [ ]:
# Replace 'N/A' salary strings with proper null
df = df.withColumn(
    'Salary',
    when(col('Salary') == 'N/A', lit(None))
    .otherwise(col('Salary').cast('double'))
)

# Impute null salaries with department-level average using Window
windowSpec = Window.partitionBy('Department')
df = df.withColumn(
    'Salary',
    when(
        col('Salary').isNull(),
        avg('Salary').over(windowSpec)
    ).otherwise(col('Salary'))
)

# Verify no nulls remain
remaining_nulls = df.filter(col('Salary').isNull()).count()
print(f'Remaining null salaries: {remaining_nulls}')
df.display()

### Data Quality Assertions

Validate Silver layer before writing to table. Pipeline fails loudly if data quality checks don't pass.

In [ ]:
# Data quality checks — assert clean data before loading to Silver
assert df.filter(col('Salary').isNull()).count() == 0, 'FAILED: Null salaries found'
assert df.filter(col('Age') <= 0).count() == 0, 'FAILED: Invalid ages found'
assert df.filter(col('Department').isNull()).count() == 0, 'FAILED: Null departments found'
assert df.filter(col('Region').isNull()).count() == 0, 'FAILED: Null regions found'
assert df.filter(col('join_date').isNull()).count() == 0, 'FAILED: Null join dates found'

print('All data quality checks passed')

### Write to Silver Delta Table

In [ ]:
# Write clean DataFrame to Silver Delta Lake table
df.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('silver_employee')

print('Silver table written successfully')
spark.sql('SELECT COUNT(*) as total_records FROM silver_employee').display()

## Gold Layer — Star Schema on Delta Lake

Creating analytical dimension and fact tables following Star Schema design. These tables support BI dashboards and ML feature pipelines.

### dim_department — Department Dimension

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW dim_department AS
SELECT DISTINCT
    DENSE_RANK() OVER (ORDER BY Department) AS department_id,
    Department
FROM silver_employee
WHERE Department IS NOT NULL
""")

spark.sql('SELECT * FROM dim_department ORDER BY department_id').display()

### dim_region — Region Dimension

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW dim_region AS
SELECT DISTINCT
    DENSE_RANK() OVER (ORDER BY Region) AS region_id,
    Region
FROM silver_employee
WHERE Region IS NOT NULL
""")

spark.sql('SELECT * FROM dim_region ORDER BY region_id').display()

### fact_salary_analytics — Central Fact Table

Joins employee records with department and region dimension keys. Stores quantitative metrics — salary and performance score — for analytical queries.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW fact_salary_analytics AS
SELECT
    s.Employee_ID,
    d.department_id,
    r.region_id,
    s.join_date,
    s.Salary,
    s.Performance_Score,
    s.remote_work
FROM silver_employee s
LEFT JOIN dim_department d ON s.Department = d.Department
LEFT JOIN dim_region r ON s.Region = r.Region
""")

spark.sql('SELECT * FROM fact_salary_analytics LIMIT 10').display()

## Sample Analytics Queries

Demonstrating the Gold layer in action — business-ready queries on the Star Schema.

In [ ]:
# Average salary by department
spark.sql("""
SELECT d.Department, ROUND(AVG(f.Salary), 2) AS avg_salary,
       COUNT(*) AS headcount
FROM fact_salary_analytics f
JOIN dim_department d ON f.department_id = d.department_id
GROUP BY d.Department
ORDER BY avg_salary DESC
""").display()

In [ ]:
# Remote work distribution by region
spark.sql("""
SELECT r.Region, f.remote_work,
       COUNT(*) AS employee_count
FROM fact_salary_analytics f
JOIN dim_region r ON f.region_id = r.region_id
GROUP BY r.Region, f.remote_work
ORDER BY r.Region, f.remote_work
""").display()

## Pipeline Summary

| Layer | Table | Description |
|---|---|---|
| Bronze | Raw CSV | Original source data, no transformations |
| Silver | silver_employee | Cleaned, typed, imputed Delta table |
| Gold | dim_department | Department dimension |
| Gold | dim_region | Region dimension |
| Gold | fact_salary_analytics | Central fact table for analytics |

**Tech Stack:** PySpark · Databricks · Delta Lake · Spark SQL · Window Functions · UDF · Star Schema · Medallion Architecture